# Test Set Data Distribution Analysis

This notebook examines the class distributions in the test sets used in `evaluate.py` for both 30-day mortality (MOR30) and 30-day readmission (REA30) outcomes.

The test data is:
- Combined from NRD 2021 and 2022
- Preprocessed with the same steps as in `evaluate.py`
- Downsampled to 10% using stratified sampling with seed=42

In [ ]:
import pandas as pd
import numpy as np
import pickle

In [ ]:
# Load encoders (needed for ICD code and AGE preprocessing)
with open('Model/full_label_encoder.pkl', 'rb') as file:
    encoder = pickle.load(file)

with open('Model/full_age_scaler.pkl', 'rb') as file:
    age_scaler = pickle.load(file)

In [ ]:
# Load and combine 2021-2022 data
file_2021 = '/users/xwang259/icd/data/NRD_2021_test.csv'
file_2022 = '/users/xwang259/icd/data/NRD_2022_test.csv'

df1 = pd.read_csv(file_2021)
df2 = pd.read_csv(file_2022)

# Combine rows from both files
combined_data = pd.concat([df1, df2], ignore_index=True)

# Convert all column names to uppercase
combined_data.columns = combined_data.columns.str.upper()

print(f"Total combined data size: {len(combined_data):,} rows")

In [ ]:
# Define stratified sampling function (same as evaluate.py)
def stratified_pick(idx, y_sub, frac, seed):
    rng = np.random.default_rng(seed)
    picked = []
    for cls in np.unique(y_sub):
        cls_idx = idx[y_sub == cls]
        n = max(1, int(np.floor(frac * len(cls_idx))))
        pick = rng.choice(cls_idx, size=n, replace=False)
        picked.append(pick)
    return np.concatenate(picked)

def preprocess_data(data, outcome_var, encoder, age_scaler):
    """
    Preprocess data exactly as done in evaluate.py
    """
    # Make a copy
    df = data.copy()
    
    # Filter out observations where DIED == 1 if outcome_var is REA30
    if 'DIED' in df.columns:
        initial_size = len(df)
        df = df[df['DIED'] != 1]
        print(f"  Removed {initial_size - len(df):,} rows where DIED==1")
    
    # Handle missing values in the target variable
    initial_size = len(df)
    df = df.dropna(subset=[outcome_var])
    print(f"  Removed {initial_size - len(df):,} rows with missing {outcome_var}")
    
    # Define ICD columns
    icd_columns = [f'I10_DX{i}' for i in range(1, 41)]
    
    # Encode ICD codes
    label_to_int = {label: idx for idx, label in enumerate(encoder.classes_)}
    unknown_label_int = encoder.transform(["NAN"])[0]
    
    for col in icd_columns:
        df[col] = df[col].astype(str).str.upper()
        df[col] = df[col].map(label_to_int).fillna(unknown_label_int).astype(int)
    
    # Normalize 'AGE'
    df['AGE'] = age_scaler.transform(df[['AGE']])
    
    # Replace -8 and -9 with NaN in PAY1 and ZIPINC_QRTL
    df['PAY1'] = df['PAY1'].replace([-8, -9], np.nan)
    df['ZIPINC_QRTL'] = df['ZIPINC_QRTL'].replace([-8, -9], np.nan)
    
    # One-hot encode 'PAY1' and 'ZIPINC_QRTL'
    df = pd.get_dummies(df, columns=['PAY1', 'ZIPINC_QRTL'], prefix=['PAY1', 'ZIPINC_QRTL'])
    
    # Get payment and income columns
    pay1_columns = [col for col in df.columns if col.startswith('PAY1_')]
    zipinc_qrtl_columns = [col for col in df.columns if col.startswith('ZIPINC_QRTL_')]
    
    # Drop rows with missing values
    X = df[['AGE', 'FEMALE'] + pay1_columns + zipinc_qrtl_columns + icd_columns]
    initial_size = len(df)
    X = X.dropna()
    df = df.loc[X.index]
    print(f"  Removed {initial_size - len(df):,} rows with missing features")
    
    return df

## 30-Day Mortality (MOR30) Test Set Distribution

In [ ]:
print("="*60)
print("Processing MOR30 (30-day Mortality)")
print("="*60)

# Preprocess for MOR30
mor30_data = preprocess_data(combined_data, 'MOR30', encoder, age_scaler)

# Get labels
y_mor30 = mor30_data['MOR30'].to_numpy()
idx_all = np.arange(len(mor30_data))

# Downsample to 10% with seed=42 (same as evaluate.py)
idx10 = stratified_pick(idx_all, y_mor30, frac=0.10, seed=42)
mor30_test = mor30_data.iloc[idx10].copy()
y_mor30_test = mor30_test['MOR30'].to_numpy()

# Count positives and negatives
n_positives_mor30 = (y_mor30_test == 1).sum()
n_negatives_mor30 = (y_mor30_test == 0).sum()
total_mor30 = len(y_mor30_test)

print(f"\nMOR30 Test Set (10% downsampled, seed=42):")
print(f"  Total samples: {total_mor30:,}")
print(f"  Positives (deaths within 30 days): {n_positives_mor30:,} ({100*n_positives_mor30/total_mor30:.2f}%)")
print(f"  Negatives (no death within 30 days): {n_negatives_mor30:,} ({100*n_negatives_mor30/total_mor30:.2f}%)")
print(f"  Positive:Negative ratio: 1:{n_negatives_mor30/n_positives_mor30:.2f}")

## 30-Day Readmission (REA30) Test Set Distribution

In [ ]:
print("="*60)
print("Processing REA30 (30-day Readmission)")
print("="*60)

# Preprocess for REA30 (this will filter out DIED==1)
rea30_data = preprocess_data(combined_data, 'REA30', encoder, age_scaler)

# Get labels
y_rea30 = rea30_data['REA30'].to_numpy()
idx_all = np.arange(len(rea30_data))

# Downsample to 10% with seed=42 (same as evaluate.py)
idx10 = stratified_pick(idx_all, y_rea30, frac=0.10, seed=42)
rea30_test = rea30_data.iloc[idx10].copy()
y_rea30_test = rea30_test['REA30'].to_numpy()

# Count positives and negatives
n_positives_rea30 = (y_rea30_test == 1).sum()
n_negatives_rea30 = (y_rea30_test == 0).sum()
total_rea30 = len(y_rea30_test)

print(f"\nREA30 Test Set (10% downsampled, seed=42):")
print(f"  Total samples: {total_rea30:,}")
print(f"  Positives (readmitted within 30 days): {n_positives_rea30:,} ({100*n_positives_rea30/total_rea30:.2f}%)")
print(f"  Negatives (no readmission within 30 days): {n_negatives_rea30:,} ({100*n_negatives_rea30/total_rea30:.2f}%)")
print(f"  Positive:Negative ratio: 1:{n_negatives_rea30/n_positives_rea30:.2f}")

## Summary Comparison

In [ ]:
# Create summary table
summary = pd.DataFrame({
    'Outcome': ['MOR30', 'REA30'],
    'Total Samples': [total_mor30, total_rea30],
    'Positives': [n_positives_mor30, n_positives_rea30],
    'Negatives': [n_negatives_mor30, n_negatives_rea30],
    'Positive %': [100*n_positives_mor30/total_mor30, 100*n_positives_rea30/total_rea30],
    'Negative %': [100*n_negatives_mor30/total_mor30, 100*n_negatives_rea30/total_rea30]
})

print("\n" + "="*80)
print("SUMMARY: Test Set Distributions (10% downsampled with seed=42)")
print("="*80)
print(summary.to_string(index=False))
print("="*80)